# 1. InMemoryCache — 개념 이해용

In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

set_llm_cache(InMemoryCache())
llm = ChatOpenAI(model="gpt-4o-mini")

message = [HumanMessage(content="서울 광장시장에서 가장 맛있는 길거리 음식은?")]

## 1.1. 처음 호출

In [2]:
%%time
# 첫 번째 호출 - API 실제 호출
response = llm.invoke(message)
print(response.content)
# Wall time: 약 2~3초

서울 광장시장에서 맛있는 길거리 음식으로는 다음과 같은 것들이 있습니다:

1. **떡볶이**: 매콤하고 쫄깃한 떡볶이는 광장시장에서 매우 인기 있는 간식입니다.
2. **튀김**: 다양한 해물과 채소를 튀겨낸 튀김이 여러 종류 있어, 바삭한 식감을 즐길 수 있습니다.
3. **오뎅 (어묵)**: 따끈하고 국물이 있는 오뎅은 겨울철에 특히 인기가 많습니다.
4. **핫도그**: 치즈나 다양한 소스를 얹은 핫도그는 간편하면서도 맛있는 간식입니다.
5. **호떡**: 달콤한 설탕과 견과류가 들어간 호떡은 따뜻하게 구워서 먹으면 아주 맛있습니다.

이 외에도 다양한 길거리 음식들이 있으니, 여러 가지를 시도해 보는 것을 추천합니다!
CPU times: total: 15.6 ms
Wall time: 4.75 s


## 1.2. 캐싱 이후 호출

In [4]:
%%time
# 두 번째 호출 - 캐시에서 즉시 반환
response = llm.invoke(message)
print(response.content)
# Wall time: 약 1ms (거의 0)

서울 광장시장에서 맛있는 길거리 음식으로는 다음과 같은 것들이 있습니다:

1. **떡볶이**: 매콤하고 쫄깃한 떡볶이는 광장시장에서 매우 인기 있는 간식입니다.
2. **튀김**: 다양한 해물과 채소를 튀겨낸 튀김이 여러 종류 있어, 바삭한 식감을 즐길 수 있습니다.
3. **오뎅 (어묵)**: 따끈하고 국물이 있는 오뎅은 겨울철에 특히 인기가 많습니다.
4. **핫도그**: 치즈나 다양한 소스를 얹은 핫도그는 간편하면서도 맛있는 간식입니다.
5. **호떡**: 달콤한 설탕과 견과류가 들어간 호떡은 따뜻하게 구워서 먹으면 아주 맛있습니다.

이 외에도 다양한 길거리 음식들이 있으니, 여러 가지를 시도해 보는 것을 추천합니다!
CPU times: total: 0 ns
Wall time: 1 ms


# 2. SQLiteCache — 로컬 영구 저장 (실습 권장)

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.globals import set_llm_cache
from langchain_community.cache import SQLiteCache

# 프로젝트 폴더에 .langchain.db 파일로 저장
set_llm_cache(SQLiteCache(database_path=".langchain.db"))

llm = ChatOpenAI(model="gpt-4o-mini")
message = [HumanMessage(content="코스피 지수란 무엇인가요? 한 문장으로 답해주세요.")]

## 2.1. 첫번째 호출

In [4]:
%%time
# 첫 번째 호출 - API 실제 호출 후 파일에 저장
response = llm.invoke(message)
print(response.content)
# Wall time: 약 2~3초

코스피 지수는 한국거래소에 상장된 기업들의 주가를 종합적으로 반영하는 주식 시장 지수입니다.
CPU times: total: 31.2 ms
Wall time: 1 s


## 2.2. 두번째 호출

In [5]:
%%time
# 두 번째 호출 - 파일에서 즉시 반환 (프로그램 재시작 후에도 동일)
response = llm.invoke(message)
print(response.content)
# Wall time: 약 2~5ms

코스피 지수는 한국거래소에 상장된 기업들의 주가를 종합적으로 반영하는 주식 시장 지수입니다.
CPU times: total: 0 ns
Wall time: 1.99 ms


- 일반 캐시 (정확 일치)   
─────────────────────────────────────────   
"서울에서 맛있는 길거리 음식" → 캐시 적중    
"서울에서 유명한 길거리 음식" → 캐시 미적중 (다른 문자열)    

- 시맨틱 캐시 (의미 유사)    
─────────────────────────────────────────      
"서울에서 맛있는 길거리 음식" → 캐시 저장     
"서울에서 유명한 길거리 음식" → 캐시 적중 (유사도 임계값 초과)    